In [1]:
# ============================================================================
# ENHANCED TRAINING AND PREDICTION PIPELINE FOR TELECOM CHURN
# ============================================================================
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, QuantileTransformer
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectFromModel
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

from utils import create_advanced_features, get_feature_columns, clean_feature_names

# ============================================================================
# LOAD AND PREPARE TRAINING DATA
# ============================================================================
print("=" * 80)
print("LOADING TRAINING DATA")
print("=" * 80)

df_train = pd.read_csv('./train.csv')
print(f"Training data shape: {df_train.shape}")
print(f"\nClass distribution:")
print(df_train['label'].value_counts())
print(f"Class ratio: {df_train['label'].value_counts()[0] / df_train['label'].value_counts()[1]:.2f}:1")

# Check for missing values
print(f"\nMissing values summary:")
missing_summary = df_train.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(missing_summary.head(10))
else:
    print("No missing values found")

# ============================================================================
# ADVANCED FEATURE ENGINEERING
# ============================================================================
print("\n" + "=" * 80)
print("CREATING ADVANCED FEATURES")
print("=" * 80)

df_train_processed = create_advanced_features(df_train, is_train=True)

# Get feature columns
feature_cols = get_feature_columns(df_train_processed)
print(f"Total features after engineering: {len(feature_cols)}")

# Clean feature names
original_feature_cols = feature_cols.copy()
cleaned_feature_cols = clean_feature_names(feature_cols)
feature_name_mapping = dict(zip(cleaned_feature_cols, original_feature_cols))

# Prepare X and y
X = df_train_processed[feature_cols].copy()
X.columns = cleaned_feature_cols
y = df_train_processed['label']

# Handle missing values
print("\nHandling missing values...")
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col].fillna(X[col].median(), inplace=True)
    else:
        X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)

# Replace infinities
X.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Final feature matrix shape: {X.shape}")
print(f"\nFeature statistics:")
print(f"  Numeric features: {X.select_dtypes(include=[np.number]).shape[1]}")
print(f"  Memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# ============================================================================
# FEATURE SCALING (Optional - for some models)
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)

# Create scaled version for models that benefit from it
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns,
    index=X.index
)
print("Features scaled using StandardScaler")

# ============================================================================
# TRAIN-VALIDATION SPLIT
# ============================================================================
print("\n" + "=" * 80)
print("SPLITTING DATA FOR VALIDATION")
print("=" * 80)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Training class distribution: {y_train.value_counts().to_dict()}")
print(f"Validation class distribution: {y_val.value_counts().to_dict()}")

LOADING TRAINING DATA
Training data shape: (59904, 88)

Class distribution:
label
0    44904
1    15000
Name: count, dtype: int64
Class ratio: 2.99:1

Missing values summary:
gm_use_dur               47451
gm_dayt_use_dur          47451
gm_ngt_use_dur           47451
shrt_vid_use_dur         47451
shrt_vid_dayt_use_dur    47451
shrt_vid_ngt_use_dur     47451
long_vid_use_dur         47451
long_vid_dayt_use_dur    47451
long_vid_ngt_use_dur     47451
anchor_use_dur           47451
dtype: int64

CREATING ADVANCED FEATURES
Total features after engineering: 208

Handling missing values...
Final feature matrix shape: (59904, 208)

Feature statistics:
  Numeric features: 208
  Memory usage: 95.06 MB

FEATURE SCALING
Features scaled using StandardScaler

SPLITTING DATA FOR VALIDATION
Training set: (47923, 208)
Validation set: (11981, 208)
Training class distribution: {0: 35923, 1: 12000}
Validation class distribution: {0: 8981, 1: 3000}


In [ ]:

# ============================================================================
# ADVANCED MODEL TRAINING WITH CROSS-VALIDATION
# ============================================================================
print("\n" + "=" * 80)
print("TRAINING ADVANCED MODELS WITH CROSS-VALIDATION")
print("=" * 80)

# Calculate scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Setup cross-validation
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

# Store results
results = {}

# ============================================================================
# MODEL 2: XGBOOST WITH ADVANCED PARAMETERS
# ============================================================================
print("\n" + "-" * 80)
print("2. TRAINING XGBOOST (REGULAR AND IMBALANCED VERSIONS)")
print("-" * 80)

# Enhanced XGBoost parameters
xgb_params_enhanced = {
    'n_estimators': 1800,
    'max_depth': 10,
    'learning_rate': 0.02,
    'subsample': 0.85,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'colsample_bynode': 0.8,
    'min_child_weight': 3,
    'gamma': 0.05,
    'reg_alpha': 0.4,
    'reg_lambda': 0.4,
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist',
    'grow_policy': 'depthwise',
    'max_bin': 512,          # Increased bins
    'eval_metric': 'auc'
}

# ============================================================================
# 2A. REGULAR XGBOOST
# ============================================================================
print("\n2A. Training Regular XGBoost...")
xgb_model = xgb.XGBClassifier(**xgb_params_enhanced)

# Cross-validation on original data
print("Performing 5-fold cross-validation on original data...")
cv_scores_xgb = cross_val_score(xgb_model, X_train, y_train, cv=skf, 
                                scoring='roc_auc', n_jobs=-1)
print(f"CV ROC-AUC scores: {cv_scores_xgb}")
print(f"Mean CV ROC-AUC: {cv_scores_xgb.mean():.4f} (+/- {cv_scores_xgb.std() * 2:.4f})")

# Train on full training set with early stopping
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Evaluate on validation set
y_pred_xgb = xgb_model.predict(X_val)
y_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]
xgb_acc = accuracy_score(y_val, y_pred_xgb)
xgb_auc = roc_auc_score(y_val, y_proba_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb)

print(f"\nRegular XGBoost Validation Metrics:")
print(f"  Accuracy: {xgb_acc:.4f}")
print(f"  ROC-AUC: {xgb_auc:.4f}")
print(f"  F1-Score: {xgb_f1:.4f}")

# ============================================================================
# STORE RESULTS
# ============================================================================
results['XGBoost'] = {
    'model': xgb_model,
    'cv_scores': cv_scores_xgb,
    'val_auc': xgb_auc,
    'val_f1': xgb_f1,
    'val_acc': xgb_acc,
    'predictions': y_pred_xgb,
    'probabilities': y_proba_xgb
}

# ============================================================================
# MODEL COMPARISON AND SELECTION
# ============================================================================
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY")
print("=" * 80)

# Define custom scoring function
def custom_score(y_true, y_pred, y_proba=None):
    """Calculate custom score: 0.7 * accuracy + 0.3 * f1"""
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return 0.7 * acc + 0.3 * f1

# Add custom scores to existing models
for model_name in results.keys():
    if 'predictions' in results[model_name] and 'val_custom_score' not in results[model_name]:
        y_pred = results[model_name]['predictions']
        results[model_name]['val_custom_score'] = custom_score(y_val, y_pred)

# Create comparison dataframe
comparison_data = []
for model_name, model_results in results.items():
    if 'cv_scores' in model_results:
        cv_mean = model_results['cv_scores'].mean()
        cv_std = model_results['cv_scores'].std()
    else:
        cv_mean = cv_std = np.nan
    
    comparison_data.append({
        'Model': model_name,
        'CV_Score_Mean': cv_mean,
        'CV_Score_Std': cv_std,
        'Val_AUC': model_results['val_auc'],
        'Val_F1': model_results['val_f1'],
        'Val_Acc': model_results.get('val_acc', np.nan),
        'Val_Custom_Score': model_results.get('val_custom_score', np.nan)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.sort_values('Val_Custom_Score', ascending=False)

print("\nModel Performance Comparison (sorted by Custom Score):")
print("=" * 90)
print(f"{'Model':<20} | {'CV Score':<15} | {'Val AUC':<8} | {'Val F1':<8} | {'Val Acc':<8} | {'Custom Score':<12}")
print("-" * 90)

for _, row in comparison_df.iterrows():
    if pd.isna(row['CV_Score_Mean']):
        cv_score_str = "N/A"
    else:
        cv_score_str = f"{row['CV_Score_Mean']:.4f}±{row['CV_Score_Std']:.4f}"
    
    print(f"{row['Model']:<20} | {cv_score_str:<15} | {row['Val_AUC']:<8.4f} | "
          f"{row['Val_F1']:<8.4f} | {row['Val_Acc']:<8.4f} | {row['Val_Custom_Score']:<12.4f}")

# Select best model based on custom score
best_model_name = comparison_df.iloc[0]['Model']
best_model_results = results[best_model_name]

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   Custom Score (0.7*Acc + 0.3*F1): {best_model_results['val_custom_score']:.4f}")
print(f"   Validation Accuracy: {best_model_results.get('val_acc', 'N/A'):.4f}")
print(f"   Validation F1: {best_model_results['val_f1']:.4f}")
print(f"   Validation AUC: {best_model_results['val_auc']:.4f}")

# Display top 3 models
print(f"\n📊 TOP 3 MODELS BY CUSTOM SCORE:")
print("-" * 50)
for i in range(min(3, len(comparison_df))):
    row = comparison_df.iloc[i]
    print(f"{i+1}. {row['Model']:<20} Score: {row['Val_Custom_Score']:.4f}")

# ============================================================================
# FEATURE IMPORTANCE ANALYSIS
# ============================================================================
print("\n" + "=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)

def get_feature_importance(model, feature_names, model_type):
    """Extract feature importance from different model types"""
    if hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importance = abs(model.coef_[0])
    else:
        return None
    
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': importance
    }).sort_values('importance', ascending=False)
    
    return feature_importance

# Get feature importance from best tree-based models
models_for_importance = ['LightGBM', 'XGBoost', 'CatBoost', 'RandomForest']

print("Top 20 Most Important Features:")
print("-" * 50)

for model_name in models_for_importance:
    if model_name in results and 'model' in results[model_name]:
        importance_df = get_feature_importance(
            results[model_name]['model'], 
            X.columns, 
            model_name
        )
        
        if importance_df is not None:
            print(f"\n{model_name}:")
            top_features = importance_df.head(10)
            for idx, row in top_features.iterrows():
                original_name = feature_name_mapping.get(row['feature'], row['feature'])
                print(f"  {row['feature'][:30]:30s} | {row['importance']:.4f}")

# ============================================================================
# SAVE MODELS AND RESULTS
# ============================================================================
print("\n" + "=" * 80)
print("SAVING MODELS AND RESULTS")
print("=" * 80)

import pickle
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# Save best model
best_model = best_model_results['model'] if 'model' in best_model_results else None
if best_model is not None:
    with open(f'models/best_model_{best_model_name.lower()}.pkl', 'wb') as f:
        pickle.dump(best_model, f)
    print(f"✅ Best model ({best_model_name}) saved to models/")

# Save feature columns and preprocessing objects
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('models/feature_name_mapping.pkl', 'wb') as f:
    pickle.dump(feature_name_mapping, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save results summary
results_summary = {
    'model_comparison': comparison_df.to_dict('records'),
    'best_model_name': best_model_name,
    'best_model_auc': best_model_results['val_auc'],
    'best_model_f1': best_model_results['val_f1'],
    'feature_count': len(feature_cols),
    'training_samples': len(X_train),
    'validation_samples': len(X_val)
}

with open('models/training_results.pkl', 'wb') as f:
    pickle.dump(results_summary, f)

print(f"✅ Feature preprocessing objects saved")
print(f"✅ Training results summary saved")

print("\n" + "=" * 80)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("=" * 80)
print(f"🎯 Best Model: {best_model_name}")
print(f"📊 Validation AUC: {best_model_results['val_auc']:.4f}")
print(f"📈 Validation F1: {best_model_results['val_f1']:.4f}")
print(f"💾 Models saved to 'models/' directory")
print("=" * 80)


In [2]:
# ============================================================================
# LOAD SAVED MODELS AND PREPROCESSING OBJECTS
# ============================================================================
print("\n" + "=" * 80)
print("LOADING SAVED MODELS AND PREPROCESSING OBJECTS")
print("=" * 80)

import pickle
import os
import pandas as pd
import numpy as np

# Check if models directory exists
if not os.path.exists('models'):
    raise FileNotFoundError("Models directory not found! Please run training first.")

# ============================================================================
# LOAD PREPROCESSING OBJECTS
# ============================================================================
print("📦 Loading preprocessing objects...")

# Load feature columns
with open('models/feature_columns.pkl', 'rb') as f:
    feature_cols = pickle.load(f)
print(f"✅ Feature columns loaded ({len(feature_cols)} features)")

# Load feature name mapping
with open('models/feature_name_mapping.pkl', 'rb') as f:
    feature_name_mapping = pickle.load(f)
print(f"✅ Feature name mapping loaded")

# Load scaler
with open('models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
print(f"✅ Scaler loaded")

# ============================================================================
# LOAD TRAINING RESULTS
# ============================================================================
print("\n📊 Loading training results...")

with open('models/training_results.pkl', 'rb') as f:
    results_summary = pickle.load(f)

best_model_name = results_summary['best_model_name']
print(f"✅ Training results loaded")
print(f"   Best model: {best_model_name}")
print(f"   Validation AUC: {results_summary['best_model_auc']:.4f}")
print(f"   Validation F1: {results_summary['best_model_f1']:.4f}")

# ============================================================================
# LOAD BEST MODEL
# ============================================================================
print(f"\n🤖 Loading best model ({best_model_name})...")

model_filename = f'models/best_model_{best_model_name.lower()}.pkl'
if os.path.exists(model_filename):
    with open(model_filename, 'rb') as f:
        best_model = pickle.load(f)
    print(f"✅ Best model loaded successfully")
else:
    raise FileNotFoundError(f"Model file {model_filename} not found!")

# ============================================================================
# DISPLAY MODEL INFORMATION
# ============================================================================
print("\n" + "=" * 80)
print("LOADED MODEL INFORMATION")
print("=" * 80)
print(f"🎯 Model Type: {best_model_name}")
print(f"📊 Performance:")
print(f"   • Validation AUC: {results_summary['best_model_auc']:.4f}")
print(f"   • Validation F1:  {results_summary['best_model_f1']:.4f}")
print(f"🔢 Data Info:")
print(f"   • Features: {results_summary['feature_count']}")
print(f"   • Training samples: {results_summary['training_samples']:,}")
print(f"   • Validation samples: {results_summary['validation_samples']:,}")
print("=" * 80)



# ============================================================================
# LOAD AND PREDICT TEST DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING TEST DATA")
print("=" * 80)

df_test = pd.read_csv('./testA.csv')
print(f"Test data shape: {df_test.shape}")

# Store IDs
test_ids = df_test['id'].copy()

# Create features
print("\nCreating features for test data...")
df_test_processed = create_advanced_features(df_test, is_train=True)

# Prepare test features
X_test = df_test_processed[feature_cols].copy()
X_test.columns = cleaned_feature_cols

# Fill missing values
print("Handling missing values in test data...")
for col in X_test.columns:
    if col in X.columns:
        if X_test[col].dtype in ['float64', 'int64']:
            X_test[col].fillna(X[col].median(), inplace=True)
        else:
            X_test[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)
    else:
        X_test[col].fillna(0, inplace=True)

# Replace infinities
X_test.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Test feature matrix shape: {X_test.shape}")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================

print("\n" + "=" * 80)
print("MAKING PREDICTIONS")
print("=" * 80)

predictions = best_model.predict(X_test)
predictions_proba = best_model.predict_proba(X_test)[:, 1]

print(f"Predictions distribution:")
print(f"Class 0: {(predictions == 0).sum()} ({(predictions == 0).sum() / len(predictions) * 100:.1f}%)")
print(f"Class 1: {(predictions == 1).sum()} ({(predictions == 1).sum() / len(predictions) * 100:.1f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

# Save to CSV
output_path = './submit.csv'
submission.to_csv(output_path, index=False)

print(f"Submission saved to: {output_path}")
print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))

# Also save with probabilities
submission_with_proba = pd.DataFrame({
    'id': test_ids,
    'label': predictions,
    'probability': predictions_proba
})

output_path_proba = './submit_with_probabilities.csv'
submission_with_proba.to_csv(output_path_proba, index=False)
print(f"\nPredictions with probabilities saved to: {output_path_proba}")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"Best Model: {best_model_name}")
# print(f"Validation ROC-AUC: {best_auc:.4f}")
print(f"Total Features Used: {len(feature_cols)}")


LOADING SAVED MODELS AND PREPROCESSING OBJECTS
📦 Loading preprocessing objects...
✅ Feature columns loaded (208 features)
✅ Feature name mapping loaded
✅ Scaler loaded

📊 Loading training results...
✅ Training results loaded
   Best model: XGBoost
   Validation AUC: 0.8467
   Validation F1: 0.6177

🤖 Loading best model (XGBoost)...
✅ Best model loaded successfully

LOADED MODEL INFORMATION
🎯 Model Type: XGBoost
📊 Performance:
   • Validation AUC: 0.8467
   • Validation F1:  0.6177
🔢 Data Info:
   • Features: 208
   • Training samples: 47,923
   • Validation samples: 11,981

LOADING TEST DATA
Test data shape: (19999, 87)

Creating features for test data...
Handling missing values in test data...
Test feature matrix shape: (19999, 208)

MAKING PREDICTIONS
Predictions distribution:
Class 0: 19980 (99.9%)
Class 1: 19 (0.1%)

SAVING RESULTS
Submission saved to: ./submit.csv
Submission shape: (19999, 2)

First 10 predictions:
                                     id  label
0  7780c4fd-8b15-4